[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Migrations


## What you will be able to do

Change a schema that already has rows in it, with `pwmigrate`: write the first migration from the
models, apply it, and read what the history table says. Ask what the database is missing compared
with the models, and turn that answer into a migration file you can read before running it. Apply
and revert, including an index as well as a column. Use the exit code of `status` as the check that
belongs in a pipeline. Read a database somebody else made with `generate_models` and `pwiz`. And
recognize the two failures that make people think the tool is broken: an argument that is taken for
a module name, and a models file Python had already imported.


## The idea

### The problem

Every notebook before this one called `create_tables` on an empty database. That is not what
changing a schema looks like once anything is stored: the table is there, it has rows, and the
change has to be applied to it without losing them.

`create_tables` will not do it. It creates tables that do not exist and leaves alone the ones that
do, so a new field on an existing model changes nothing at all, in silence. The model says the
column is there and the database has never heard of it, and the first query that mentions it fails.

### What a migration is

A file holding two functions: `up`, which makes a change, and `down`, which takes it back. They are
numbered, they run in order, and the database keeps a table of which ones have been run so that no
migration runs twice and a new checkout can catch up.

`pwmigrate` writes them, runs them and reverts them. It can also compare the models with the
database and generate the file from the difference, which is the part that makes it worth using
rather than writing `ALTER TABLE` by hand.

### Why it works that way

The models are a description of the schema you want. The database is the schema you have. A
migration is one step from one to the other, and the reason it is a file rather than a command is
that everybody's database has to take the same steps in the same order: yours, the other
developer's, the test database and the one with the real rows in it.

### Where this shows up

The second week of any project. Adding a column, adding an index, renaming something, splitting a
table. Also the first time a colleague pulls your branch and their database does not match their
models.

### What this notebook covers

A project on disk, since none of this works against a database in memory. The first migration. The
schema difference, and generating a migration from it. Applying and reverting, with a column and an
index. The exit code worth putting in a pipeline. Reading a database that has no models file.
Then the four failures, including a `down` that is run for the first time when it is needed.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
from pathlib import Path

Path("models.py").write_text(
    "from peewee import CharField, Model, SqliteDatabase\n"
    "\n"
    "db = SqliteDatabase('app.db')\n"
    "\n"
    "class Note(Model):\n"
    "    title = CharField()\n"
    "\n"
    "    class Meta:\n"
    "        database = db\n")
Path("app.db").touch()                          # the file has to exist before pwmigrate sees it


def run(command):                               # run one pwmigrate command, give back its output
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return (done.stdout + done.stderr).strip()


print("initial:", run("pwmigrate app.db initial models"))
print("up:     ", run("pwmigrate app.db up"))

Path("models.py").write_text(Path("models.py").read_text().replace(
    "    title = CharField()\n", "    title = CharField()\n    body = CharField(null=True)\n"))

print("diff:   ", run("pwmigrate app.db diff models"))
```

```
initial: migrations/0001_initial.py
up:      applied: 0001_initial
diff:    add column note.body
```

A models file, a migration written from it, that migration applied, then one more field in the
models file. `diff` says what the database is missing, in the words you would use to say it. That
sentence is what the next command turns into a file.


## Setup

Nine imports, peewee installed and pinned, a project directory, and three helpers.

- `peewee` is the library, and `SqliteDatabase` opens the project's database for the last section
- `generate_models` and `print_model`, from `playhouse.reflection`, read a schema back out of a
  database that has no models file
- `subprocess` runs the migration tool the way you would run it in a terminal, and `sys` names this
  notebook's own Python so the tool that runs is the one just installed
- `tempfile` and `Path` make and write the project, and `re` takes the timestamps out of what the
  tool prints, so that this notebook reads the same on every run
- `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is not that

`WORK` is a throwaway directory holding `models.py`, `app.db` and, shortly, a `migrations` folder.
A database in memory would be no use here: migrations are about a database that outlives the process
that changed it.

`PWMIGRATE` is `python -m playhouse.migrations`, which is exactly what the `pwmigrate` command
installed with peewee runs. The module form is used here so that the notebook does not depend on
where a script directory ended up on your PATH. Every command below is written as you would type it,
with `pwmigrate` at the front.


In [1]:
import re
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import SqliteDatabase
from playhouse.reflection import generate_models, print_model

WORK = Path(tempfile.mkdtemp(prefix="catalog-"))                    # the project, for this session only
PWMIGRATE = f"{sys.executable} -m playhouse.migrations"             # the same tool the pwmigrate script runs

FIRST = """from peewee import CharField, ForeignKeyField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase("app.db", pragmas={"foreign_keys": 1})


class Author(Model):
    name = CharField(max_length=60, unique=True)

    class Meta:
        database = db


class Note(Model):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="notes")

    class Meta:
        database = db
"""


def run(command):
    """Run a pwmigrate command in the project and print what it printed, without the clock."""
    done = subprocess.run(f"{PWMIGRATE} {command}", shell=True, cwd=WORK,
                          capture_output=True, text=True)
    printed = (done.stdout + done.stderr).strip()
    print(re.sub(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", "<applied at>", printed) or "(no output)")
    return done.returncode


def show(name):
    """Print a file from the project, with the generator's date taken out."""
    text = (WORK / name).read_text()
    print(re.sub(r"on \d{4}-\d{2}-\d{2} \d{2}:\d{2}", "on <when>", text).rstrip())


def write_models(text):
    """Replace models.py, which is what a schema change looks like before any migration exists."""
    (WORK / "models.py").write_text(text)


write_models(FIRST)
(WORK / "app.db").touch()                                           # pwmigrate needs the file to exist

print("peewee", peewee.__version__)
print("the project holds:", sorted(path.name for path in WORK.iterdir()))


peewee 4.5.1
the project holds: ['app.db', 'models.py']


## Worked examples

### The first migration

`initial` reads the models and writes the migration that would build them from nothing:


In [2]:
run("app.db initial models")
print()
def holds(where):
    """What is in a directory, leaving out the cache Python writes when it imports models.py."""
    return sorted(path.name for path in where.iterdir() if path.name != "__pycache__")


print("the project now holds:", holds(WORK))
print("migrations:", holds(WORK / "migrations"))


migrations/0001_initial.py

the project now holds: ['app.db', 'migrations', 'models.py']
migrations: ['0001_initial.py']


`skipped: Base (no fields)` would appear here for a base class with nothing on it, which is worth
knowing before it looks like a warning. Here there is no base class, so both models were written:


In [3]:
show("migrations/0001_initial.py")


# Generated from a schema diff on <when>.
from peewee import *

def up(migrator, db):
    class Author(Model):
        name = CharField(unique=True, max_length=60)
        class Meta:
            database = db
            table_name = 'author'
    db.create_tables([Author])

    class Note(Model):
        title = CharField(max_length=80)
        author = ForeignKeyField(Author)
        class Meta:
            database = db
            table_name = 'note'
    db.create_tables([Note])


def down(migrator, db):
    migrator.migrate(migrator.drop_table('note'))
    migrator.migrate(migrator.drop_table('author'))


Two functions, and a file you can read before anything happens to the database. `up` creates the
tables, in an order that puts `author` before the `note` that points at it. `down` drops them in the
other order.

Writing the file is not applying it. `up` does that:


In [4]:
run("app.db up")
print()
run("app.db status")


applied: 0001_initial

[x] 0001_initial  <applied at>


0

The `[x]` is the history table, which the tool made in the database when it first needed it. That is
how a second `up` knows there is nothing to do:


In [5]:
print("a second up:")
run("app.db up")


a second up:
nothing to do.


0

### What the models say and what the database has

Now a change: a `body` column, and a `filed` column with an index on it.


In [6]:
write_models(FIRST.replace(
    '    author = ForeignKeyField(Author, backref="notes")\n',
    '    author = ForeignKeyField(Author, backref="notes")\n'
    "    body = CharField(max_length=200, null=True)\n"
    "    filed = IntegerField(index=True, null=True)\n"))

print("diff:")
run("app.db diff models")


diff:
add column note.body
add column note.filed
add index note (filed)


0

Three lines, one per change, and the index is named separately from the column it is on. Nothing has
happened to the database: `diff` only looks.

### From the difference to a file

`generate` writes the migration that `diff` described, and takes a name for it:


In [7]:
run("app.db generate add_body models")
print()
show("migrations/0002_add_body.py")


migrations/0002_add_body.py

# Generated from a schema diff on <when>.
from peewee import *

def up(migrator, db):
    migrator.migrate(migrator.add_column('note', 'body', CharField(null=True, max_length=200)))
    migrator.migrate(migrator.add_column('note', 'filed', IntegerField(index=True, null=True)))


def down(migrator, db):
    migrator.migrate(migrator.drop_index('note', 'note_filed'))
    migrator.migrate(migrator.drop_column('note', 'filed'))
    migrator.migrate(migrator.drop_column('note', 'body'))


`up` adds the two columns. `down` drops the index first and then the columns, in the reverse order,
which is what makes the pair symmetrical. Underneath, `migrator.add_column` and its neighbors are
the older `playhouse.migrate` API that peewee has always had, so a generated migration is a
readable version of the migration somebody used to write by hand.

Applying it, and asking again:


In [8]:
run("app.db up")
print()
print("diff now:")
run("app.db diff models")


applied: 0002_add_body

diff now:
schema matches models.


0

`schema matches models.` is the sentence worth remembering: the description and the thing described
now agree.

### Going back

`down` reverts the most recent migration:


In [9]:
run("app.db down")
print()
run("app.db status")


reverted: 0002_add_body

[x] 0001_initial  <applied at>
[ ] 0002_add_body


1

The `[ ]` is a migration that exists and has not been applied. The file is still there, so it can be
applied again, which is what makes `down` a step rather than a deletion:


In [10]:
run("app.db up")
run("app.db status")


applied: 0002_add_body
[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>


0

### The check for a pipeline

`status` exits non zero when anything is pending, which is the whole check:


In [11]:
print("everything applied -> exit", run("app.db status"))

write_models((WORK / "models.py").read_text().replace(
    "    filed = IntegerField(index=True, null=True)\n",
    "    filed = IntegerField(index=True, null=True)\n"
    "    seen = IntegerField(null=True)\n"))
run("app.db generate add_seen models")
print()
print("one generated, not applied -> exit", run("app.db status"))


[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
everything applied -> exit 0
migrations/0003_add_seen.py

[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
[ ] 0003_add_seen
one generated, not applied -> exit 1


A one at the end of a build is a database that has not caught up with the code about to be deployed
against it. Note which command answers this: `diff` prints drift and exits zero either way, so it is
for reading rather than for gating.

Applying it puts the exit code back:


In [12]:
run("app.db up")
print("exit", run("app.db status"))


applied: 0003_add_seen
[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
[x] 0003_add_seen  <applied at>
exit 0


### A database with no models file

`generate_models` builds models by reading the database, which is how you start on a schema somebody
else made:


In [13]:
existing = SqliteDatabase(str(WORK / "app.db"))
models = generate_models(existing)

print("tables found:", sorted(models))
print()
print_model(models["note"])


tables found: ['author', 'note', 'schema_migration']

note
  id AUTO PK
  title VARCHAR
  author INT FK: author.id
  body VARCHAR
  filed INT
  seen INT

index(es)
  author_id
  filed


`schema_migration` in that list is the tool's own history table, which is a good reminder that it is
an ordinary table in your database rather than something kept elsewhere.

`pwiz` is the same thing as a command, printing a models file to standard output, which is the usual
way to start: `python -m pwiz -e sqlite app.db > models.py`.

### When to reach for which

| What you want | The command |
|---|---|
| the first migration, from models, on an empty database | `pwmigrate app.db initial models` |
| what the database is missing | `pwmigrate app.db diff models` |
| a migration written from that difference | `pwmigrate app.db generate <name> models` |
| an empty migration to fill in yourself | `pwmigrate app.db create <name>` |
| to apply everything pending | `pwmigrate app.db up` |
| to take back the most recent one | `pwmigrate app.db down` |
| what has run and what has not | `pwmigrate app.db status` |
| to gate a deployment | `pwmigrate app.db status`, on its exit code |
| to record a migration without running it | `pwmigrate app.db fake` |
| models for a database you inherited | `generate_models(db)`, or `python -m pwiz` |

`generate` is the default for a change the models describe. `create` is for anything the diff cannot
see, which is most data changes: backfilling a column, splitting a name into two, or anything where
the new shape has to be filled in from the old rows.

### A change with data in it, finished

The whole cycle on a table that already has rows, which is the only case that matters.


In [14]:
sys.path.insert(0, str(WORK))
import models as project                                            # the models, as the project sees them

project.db.init(str(WORK / "app.db"), pragmas={"foreign_keys": 1})
author = project.Author.create(name="Ines O'Brien")
for title in ("A Careful Fire", "The Long Field"):
    project.Note.create(title=title, author=author)
print("rows before the change:", project.Note.select().count())

write_models((WORK / "models.py").read_text().replace(
    "    seen = IntegerField(null=True)\n",
    "    seen = IntegerField(null=True)\n"
    "    shelf = CharField(max_length=10, null=True)\n"))
run("app.db generate add_shelf models")
run("app.db up")
print()
print("rows after:", project.Note.select().count())
print("the new column:", [f.name for f in generate_models(
    SqliteDatabase(str(WORK / "app.db")))["note"]._meta.sorted_fields])


rows before the change: 2
migrations/0004_add_shelf.py
applied: 0004_add_shelf

rows after: 2
the new column: ['id', 'title', 'author', 'body', 'filed', 'seen', 'shelf']


Two rows before and two rows after, with a column that was not there when they were written. That is
the entire point of a migration, and it is the thing `create_tables` cannot do.

### Where each part came from

| In the change | What it relies on | The section that showed it |
|---|---|---|
| `generate add_shelf models` | a migration written from the difference | From the difference to a file |
| `up` | the change applied, once | The first migration |
| the rows surviving | `ALTER TABLE` rather than a new table | The idea |
| `generate_models(...)` to check | a schema read back from the database | A database with no models file |
| `status` afterwards | the history table saying what has run | The check for a pipeline |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/10-migrations-solutions.ipynb).

**1.** Print the status of the project's migrations, and say which have run.


In [15]:
# your code here


**2.** Add a `pinned` field to `Note`, print the difference, and generate the migration without
applying it.


In [16]:
# your code here


**3.** Read the generated file and say which line of it would be undone by which line of its `down`.


In [17]:
# your code here


**4.** Apply it, confirm the difference is empty, then revert it and confirm the difference is back.


In [18]:
# your code here


**5.** Show the exit code of `status` with something pending and with nothing pending.


In [19]:
# your code here


**6.** Build models from the database with `generate_models` and print the columns of `note`,
without reading `models.py`.


In [20]:
# your code here


## Common errors

### error: cannot import "app": No module named 'app'


In [21]:
elsewhere = Path(tempfile.mkdtemp(prefix="wrong-"))                 # a directory with no app.db in it
done = subprocess.run(f"{PWMIGRATE} app.db status", shell=True, cwd=elsewhere,
                      capture_output=True, text=True)

print((done.stdout + done.stderr).strip())
print("exit", done.returncode)


error: cannot import "app": No module named 'app'
exit 2


There is no file called `app.db` in that directory, and the message does not say so. The first
argument can be a path to a SQLite file, a database URL, or a dotted path to a `Database` object in
your code, and they are tried in that order. When nothing is at the path, the last of those is what
is left, so `app.db` is read as the attribute `db` of a module called `app`, and the error is about
importing.

Two things produce it and neither looks like this message: a typo in the filename, and running from
the wrong directory. The fix is to check where you are, which the message will never tell you:


In [22]:
print("here:", holds(elsewhere) or "(empty)")
print("the project:", holds(WORK))
print()
print("from the right directory:")
run("app.db status")


here: (empty)
the project: ['app.db', 'migrations', 'models.py']

from the right directory:
[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
[x] 0003_add_seen  <applied at>
[x] 0004_add_shelf  <applied at>


0

### No error, and a difference that does not change: a models file Python has already imported


In [23]:
write_models((WORK / "models.py").read_text().replace(
    "    shelf = CharField(max_length=10, null=True)\n",
    "    shelf = CharField(max_length=10, null=True)\n"
    "    archived = IntegerField(null=True)\n"))

print("on disk, Note now has archived:", "archived" in (WORK / "models.py").read_text())
print("the imported module says:", [f.name for f in project.Note._meta.sorted_fields])


on disk, Note now has archived: True
the imported module says: ['id', 'title', 'author', 'body', 'filed', 'seen']


The file has a field the module does not. Python imports a module once and keeps it, so everything in
this session that asked `models` what the schema should be got the answer from before the edit.

The tool run as a command is a new process each time and is not affected, which is what makes this
confusing: the same question gives two answers depending on who is asking. `importlib.reload` is the
answer inside a session, and restarting the runtime is the answer when reloading is not enough:


In [24]:
import importlib

project = importlib.reload(project)
print("after reload:", [f.name for f in project.Note._meta.sorted_fields])
print()
print("and the tool, which never had the problem:")
run("app.db diff models")


after reload: ['id', 'title', 'author', 'body', 'filed', 'seen', 'shelf', 'archived']

and the tool, which never had the problem:
add column note.archived


0

### error: no such column: ""blurbs""


In [25]:
run("app.db create fix_by_hand")
skeleton = sorted((WORK / "migrations").glob("*_fix_by_hand.py"))[-1]
skeleton.write_text(
    "from peewee import *\n"
    "\n"
    "\n"
    "def up(migrator, db):\n"
    "    migrator.migrate(migrator.add_column('note', 'blurb', CharField(null=True)))\n"
    "\n"
    "\n"
    "def down(migrator, db):\n"
    "    migrator.migrate(migrator.drop_column('note', 'blurbs'))      # the wrong name\n")

print("filled in:", skeleton.name)
print("up:")
run("app.db up")
print()
print("down, months later:")
print("exit", run("app.db down"))


migrations/0005_fix_by_hand.py
filled in: 0005_fix_by_hand.py
up:
applied: 0005_fix_by_hand

down, months later:
error: no such column: ""blurbs""
exit 2


The `up` was run the day it was written and worked. The `down` was written the same day, never run,
and is wrong. It is found in the one situation nobody wants to be in: something has gone wrong in
production and this is the command that was supposed to undo it.

Worse than the error is what it leaves behind. The revert failed partway, so the migration is still
recorded as applied while its work is half undone:


In [26]:
run("app.db status")
print()
print("the column the up() added is still there:",
      "blurb" in [f.name for f in generate_models(
          SqliteDatabase(str(WORK / "app.db")))["note"]._meta.sorted_fields])


[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
[x] 0003_add_seen  <applied at>
[x] 0004_add_shelf  <applied at>
[x] 0005_fix_by_hand  <applied at>

the column the up() added is still there: True


Run every `down` once, on a copy, the day you write it. `up` then `down` then `up` takes a minute and
is the only thing that turns a `down` from a hope into a step.

### peewee.OperationalError: duplicate column name: blurb


In [27]:
from playhouse.migrate import SqliteMigrator, migrate

project_db = SqliteDatabase(str(WORK / "app.db"))
migrator = SqliteMigrator(project_db)
migrate(migrator.add_column("note", "blurb", peewee.CharField(null=True)))


OperationalError: duplicate column name: blurb

The column is there, because the migration above added it. This is what running a change outside the
migration history looks like: `playhouse.migrate` is the API underneath everything in this notebook,
and used directly it has no record of what has already been done, so nothing stops it being run
twice.

That is the argument for the history table rather than a folder of scripts somebody runs by hand.
The same change through `up` is skipped, because the tool knows:


In [28]:
run("app.db up")
print("exit", run("app.db status"))


nothing to do.
[x] 0001_initial  <applied at>
[x] 0002_add_body  <applied at>
[x] 0003_add_seen  <applied at>
[x] 0004_add_shelf  <applied at>
[x] 0005_fix_by_hand  <applied at>
exit 0


## Recap

- `create_tables` makes tables that are missing and never changes one that exists, so it cannot
  apply a schema change to a database with rows in it.
- A migration is a numbered file with `up` and `down`, and a table in the database recording which
  have run.
- `initial` writes the first one from the models. `diff` prints what the database is missing.
  `generate` turns that difference into a file. `create` makes an empty one for changes a diff
  cannot see, which is most changes to data.
- `up` applies, `down` reverts one step, `status` lists what has run.
- `status` exits non zero when anything is pending, which is the check to put in a pipeline. `diff`
  exits zero either way.
- The first argument is a file path, a URL, or a dotted path, tried in that order, so a wrong path
  ends up as an import error about a module nobody wrote.
- A models module edited during a session keeps its old fields until it is reloaded, while the
  command line tool, being a new process, always sees the file.
- A `down` that has never been run is not a step, it is a guess. Run it once the day you write it.
- `generate_models` and `pwiz` build models from a database that has no models file.


## What is next

The **SQLite and PostgreSQL** notebook keeps one models module and changes the database under it:
what the same model compiles to on each backend, which field types and which clauses differ, and
what a `Database` object built from a URL changes about the code above it.


---

&#8592; **Previous:** [FTS5Model and SearchField](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/09-fts5model-and-searchfield.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [SQLite and PostgreSQL](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/11-sqlite-and-postgresql.ipynb) &#8594;
